<div style="
background:linear-gradient(135deg,#171922,#10131b);
border:2px solid #3ce7e7;
border-radius:22px;
padding:32px;
font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;
color:white;
box-shadow:0 0 18px rgba(60,231,231,.18);
">

<div style="display:flex;align-items:center;gap:14px;">
<span style="font-size:34px;">👨‍💻</span>

<div>
<div style="
font-size:34px;
font-weight:800;
color:#4ff5f5;
text-shadow:0 0 8px rgba(79,245,245,.35);
">
Aliakbar Isapour
</div>

<div style="
color:#bfbfbf;
font-size:16px;
margin-top:3px;
">
Machine Learning • Data Science • AI
</div>
</div>
</div>

<div style="
height:2px;
background:linear-gradient(90deg,#7b3ff2,#4ff5f5);
border-radius:5px;
margin:22px 0;
"></div>

<div style="
color:#d7d7d7;
font-size:16px;
line-height:1.7;
margin-bottom:25px;
">

</div>


<div style="display:flex;gap:14px;flex-wrap:wrap;">

<a href="https://github.com/isapour0"
target="_blank"
rel="noopener noreferrer"
style="
display:inline-block;
margin: 0 10px;
text-decoration:none;
background:#23283a;
color:#4ff5f5;
padding:12px 22px;
border-radius:12px;
border:1px solid #4ff5f5;
font-weight:bold;
transition:0.25s;
">
GitHub
</a>

<a href="https://www.linkedin.com/in/aliakbar-isapour-682592413"
target="_blank"
rel="noopener noreferrer"
style="
display:inline-block;
margin: 0 10px;
text-decoration:none;
background:#23283a;
color:#4ff5f5;
padding:12px 22px;
border-radius:12px;
border:1px solid #4ff5f5;
font-weight:bold;
transition:0.25s;
">
LinkedIn
</a>


</div>

</div>

# Adult Census Income: Predicting Income Level

## Problem
Given demographic and work information about a person from 1994 US Census data,
predict whether their annual income is above $50K. This is a binary classification task.

## Dataset
- This data was extracted from the 1994 Census bureau database by Ronny Kohavi and Barry Becker (Data Mining and Visualization, Silicon Graphics).
- File: `data/raw/adult.csv` (not tracked by git)
- Target: `income` (`<=50K` or `>50K`)
- Features: 14 columns, for example `age`, `workclass`, `education`, `occupation`, `hours.per.week`
- link dataset: https://www.kaggle.com/datasets/uciml/adult-census-income/data

## Approach
Compare Logistic Regression, K-Nearest Neigbors, Random Forest and XGBoost against a majority-class baseline.
Use stratified cross-validation on the training set only; the test set is opened once at the end.

## Main metric
PR-AUC (average precision), reported together with ROC-AUC and F1.
The high-income class is the minority class, so accuracy alone can look good even for a
model that is not useful. Accuracy is only reported next to the majority-class baseline.

<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-left: 8px solid #00E5FF;
border-radius:18px;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">

<h2 style="color:#00E5FF;">📌 Library Imports</h2>
<hr style="border:1px solid #8A2BE2; opacity:.4;">

<ul>
<li>Import libraries Required</li>
</ul>

</div>


In [1]:
# ---- core ----
import numpy as np
import pandas as pd
import time
import joblib
import warnings
from pathlib import Path

# ---- Visualization ----
import matplotlib.pyplot as plt
import seaborn as sns

# ---- General Prerequisite ----
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score, cross_val_predict, RandomizedSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import loguniform, randint, uniform

# ---- models ----
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ---- metrics ----
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score, roc_auc_score, precision_score, recall_score, f1_score

# ---- Config ----
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/adult.csv")   # notebook lives in notebooks/
TARGET = "income"

sns.set_theme(style="darkgrid")
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")

<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-radius:18px;
border-left: 8px solid #00E5FF;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">
    
<h2 style="color:#00E5FF;">📂 Data Loding & Initial review</h2>
<hr style="border:1px solid #8A2BE2;opacity:.4;">
</div>
